In [ ]:
# 自动安装缺失的依赖包（如果还没有安装）
import sys
import subprocess

try:
    from hmmlearn import hmm
    print("✅ hmmlearn 已安装")
except ImportError:
    print("📦 正在安装 hmmlearn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hmmlearn", "-q"])
    from hmmlearn import hmm
    print("✅ hmmlearn 安装完成！")

# Cross-Asset Momentum Regime Classifier MVP

This notebook implements a complete MVP for cross-asset momentum trading with regime-based dynamic allocation.

## Methodology
1. **Data Collection**: Equity indices (SPY/QQQ/IWM), FX pairs, macro indicators (VIX/yield/DXY)
2. **Signal Calculation**: 12-month momentum for equity, FX carry signals
3. **Feature Engineering**: VIX z-score, yield curve, DXY momentum, cross-asset correlation
4. **Regime Detection**: Gaussian HMM with 2 states
5. **Backtesting**: Compare static vs dynamic regime-based strategies


In [1]:
# Import libraries
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler
import warnings
import yaml
from pathlib import Path
from tqdm import tqdm
import time

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load config
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Libraries loaded successfully!")
print(f"Data range: {config['data']['start_date']} to {config['data']['end_date']}")


ModuleNotFoundError: No module named 'hmmlearn'

## 1. Data Collection

Fetch equity indices, FX pairs, and macro indicators from Yahoo Finance.

In [ ]:
def load_all_data(start_date, end_date, equity_symbols, fx_symbols, macro_symbols):
    """
    Load all market data from Yahoo Finance with progress visualization.
    
    Returns:
        dict: Dictionary with 'equity', 'fx', 'macro' DataFrames
    """
    print("="*60)
    print("📊 Fetching data from Yahoo Finance...")
    print("="*60)
    
    all_data = {}
    all_symbols = []
    all_symbols.extend([('equity', s) for s in equity_symbols])
    all_symbols.extend([('fx', s) for s in fx_symbols])
    all_symbols.extend([('macro', s) for s in macro_symbols])
    
    total_symbols = len(all_symbols)
    
    # Initialize data dictionaries
    equity_data = {}
    fx_data = {}
    macro_data = {}
    
    # Fetch all data with progress bar
    print(f"\n📥 Downloading {total_symbols} assets...\n")
    with tqdm(total=total_symbols, desc="Overall Progress", unit="asset", 
              bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]') as pbar:
        
        # Fetch equity data
        pbar.set_description("📈 Equity Indices")
        for symbol in equity_symbols:
            try:
                start_time = time.time()
                ticker = yf.Ticker(symbol)
                df = ticker.history(start=start_date, end=end_date)
                equity_data[symbol] = df['Close']
                elapsed = time.time() - start_time
                pbar.set_postfix_str(f"✓ {symbol} ({len(df)} days, {elapsed:.1f}s)")
                pbar.update(1)
            except Exception as e:
                pbar.set_postfix_str(f"✗ {symbol}: Error - {str(e)[:30]}")
                pbar.update(1)
        
        # Fetch FX data
        pbar.set_description("💱 FX Pairs")
        for symbol in fx_symbols:
            try:
                start_time = time.time()
                ticker = yf.Ticker(symbol)
                df = ticker.history(start=start_date, end=end_date)
                fx_data[symbol] = df['Close']
                elapsed = time.time() - start_time
                pbar.set_postfix_str(f"✓ {symbol} ({len(df)} days, {elapsed:.1f}s)")
                pbar.update(1)
            except Exception as e:
                pbar.set_postfix_str(f"✗ {symbol}: Error - {str(e)[:30]}")
                pbar.update(1)
        
        # Fetch macro data
        pbar.set_description("📊 Macro Indicators")
        for symbol in macro_symbols:
            try:
                start_time = time.time()
                ticker = yf.Ticker(symbol)
                df = ticker.history(start=start_date, end=end_date)
                macro_data[symbol] = df['Close']
                elapsed = time.time() - start_time
                pbar.set_postfix_str(f"✓ {symbol} ({len(df)} days, {elapsed:.1f}s)")
                pbar.update(1)
            except Exception as e:
                pbar.set_postfix_str(f"✗ {symbol}: Error - {str(e)[:30]}")
                pbar.update(1)
    
    # Create DataFrames
    print("\n🔄 Processing data...")
    print(f"   Equity assets loaded: {len(equity_data)}")
    print(f"   FX assets loaded: {len(fx_data)}")
    print(f"   Macro assets loaded: {len(macro_data)}")
    
    equity_df = pd.DataFrame(equity_data)
    fx_df = pd.DataFrame(fx_data)
    macro_df = pd.DataFrame(macro_data)
    
    print(f"   Equity DataFrame shape: {equity_df.shape}")
    print(f"   FX DataFrame shape: {fx_df.shape}")
    print(f"   Macro DataFrame shape: {macro_df.shape}")
    
    all_data['equity'] = equity_df
    all_data['fx'] = fx_df
    all_data['macro'] = macro_df
    
    # Combine all data
    print("🔗 Combining datasets...")
    combined = pd.concat([equity_df, fx_df, macro_df], axis=1)
    combined = combined.sort_index()
    
    # Forward fill missing values (handle market holidays)
    combined = combined.ffill()
    
    # Drop rows where ALL values are NaN (more lenient than dropna())
    # This allows some assets to have missing data while keeping the rest
    combined = combined.dropna(how='all')
    
    # For remaining NaN values, use backward fill as well
    combined = combined.bfill()
    
    # Finally, drop rows that still have any NaN (only if critical)
    # But first check if we have enough data
    if len(combined.dropna()) > 0:
        combined = combined.dropna()
    else:
        # If dropna removes everything, be more lenient - only drop rows with >50% NaN
        threshold = len(combined.columns) * 0.5
        combined = combined.dropna(thresh=threshold)
    
    print("\n" + "="*60)
    print("✅ Data loading complete!")
    print("="*60)
    if len(combined) > 0:
        print(f"📅 Date range: {combined.index[0].date()} to {combined.index[-1].date()}")
        print(f"📊 Total days: {len(combined)}")
    else:
        print("⚠️  WARNING: No overlapping data found after combining datasets!")
        print("   This may indicate missing data or date range issues.")
    print(f"📈 Assets loaded: {len(combined.columns)}")
    print(f"   - Equity: {len(equity_df.columns)}")
    print(f"   - FX: {len(fx_df.columns)}")
    print(f"   - Macro: {len(macro_df.columns)}")
    print("="*60 + "\n")
    
    return all_data, combined

# Load data
data_dict, combined_df = load_all_data(
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date'],
    equity_symbols=config['data']['equity']['symbols'],
    fx_symbols=config['data']['fx']['symbols'],
    macro_symbols=config['data']['macro']['symbols']
)

combined_df.head()


## 2. Signal Calculation

Calculate momentum signals for equity and carry signals for FX.

In [ ]:
def calculate_momentum_signals(prices_df, lookback_days=252, lag_days=21):
    """
    Calculate momentum signals: 12-month return with 1-month lag.
    
    Returns:
        DataFrame with momentum signals
    """
    signals = pd.DataFrame(index=prices_df.index)
    
    for col in prices_df.columns:
        # Calculate returns
        returns = prices_df[col].pct_change()
        
        # Calculate rolling return over lookback period
        rolling_return = prices_df[col].pct_change(lookback_days)
        
        # Lag by lag_days
        signal = rolling_return.shift(lag_days)
        
        signals[f'{col}_momentum'] = signal
    
    return signals

def calculate_portfolio_signals(equity_df, fx_df, lookback_days=252, lag_days=21):
    """
    Calculate portfolio-level signals:
    - Equity momentum: equal-weight SPY/QQQ/IWM momentum
    - FX carry: equal-weight FX momentum (proxy for carry)
    """
    # Equity momentum signals
    equity_signals = calculate_momentum_signals(equity_df, lookback_days, lag_days)
    equity_momentum = equity_signals.mean(axis=1)
    
    # FX carry signals (using momentum as proxy)
    fx_signals = calculate_momentum_signals(fx_df, lookback_days, lag_days)
    fx_carry = fx_signals.mean(axis=1)
    
    # Combine into signals DataFrame
    signals_df = pd.DataFrame({
        'equity_momentum': equity_momentum,
        'fx_carry': fx_carry
    }, index=equity_df.index)
    
    return signals_df

# Calculate signals
signals_df = calculate_portfolio_signals(
    data_dict['equity'],
    data_dict['fx'],
    lookback_days=config['signals']['momentum']['lookback_days'],
    lag_days=config['signals']['momentum']['lag_days']
)

print("Signal calculation complete!")
print(f"\nSignals shape: {signals_df.shape}")
if len(signals_df) > 0:
    print(f"Date range: {signals_df.index[0].date()} to {signals_df.index[-1].date()}")
    print(f"\nSignal statistics:")
    print(signals_df.describe())
else:
    print("⚠️  WARNING: No signals calculated - check if data was loaded correctly!")
    print(f"   Equity data shape: {data_dict['equity'].shape}")
    print(f"   FX data shape: {data_dict['fx'].shape}")

# Plot signals
if len(signals_df) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    axes[0].plot(signals_df.index, signals_df['equity_momentum'], label='Equity Momentum', linewidth=1.5)
    axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    axes[0].set_ylabel('Signal Value', fontsize=12)
    axes[0].set_title('Equity Momentum Signal (12-month return, 1-month lag)', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(signals_df.index, signals_df['fx_carry'], label='FX Carry', color='orange', linewidth=1.5)
    axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('Signal Value', fontsize=12)
    axes[1].set_xlabel('Date', fontsize=12)
    axes[1].set_title('FX Carry Signal (12-month return, 1-month lag)', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/signals.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️  Skipping plot - no signal data available")


## 3. Feature Engineering

Create features for regime detection: VIX z-score, yield curve slope, DXY momentum, cross-asset correlation.

In [ ]:
def engineer_features(combined_df, signals_df, rolling_window=60):
    """
    Engineer features for regime detection:
    1. VIX level (z-score, rolling)
    2. Yield curve slope (10Y - 2Y, z-score)
    3. DXY momentum (12-month return, z-score)
    4. Cross-asset correlation (equity vs FX, rolling)
    5. Equity momentum signal (raw)
    6. FX carry signal (raw)
    """
    features = pd.DataFrame(index=combined_df.index)
    
    # 1. VIX z-score (60-day rolling)
    if '^VIX' in combined_df.columns:
        vix = combined_df['^VIX']
        vix_mean = vix.rolling(window=rolling_window).mean()
        vix_std = vix.rolling(window=rolling_window).std()
        features['vix_zscore'] = (vix - vix_mean) / vix_std
    
    # 2. Yield curve slope (10Y - 3M, z-score)
    if '^TNX' in combined_df.columns and '^IRX' in combined_df.columns:
        yield_10y = combined_df['^TNX']
        yield_3m = combined_df['^IRX']
        yield_spread = yield_10y - yield_3m
        spread_mean = yield_spread.rolling(window=rolling_window).mean()
        spread_std = yield_spread.rolling(window=rolling_window).std()
        features['yield_spread_zscore'] = (yield_spread - spread_mean) / spread_std
    
    # 3. DXY momentum (12-month return, z-score)
    if 'DX-Y.NYB' in combined_df.columns:
        dxy = combined_df['DX-Y.NYB']
        dxy_momentum = dxy.pct_change(252)  # 12-month return
        dxy_mean = dxy_momentum.rolling(window=rolling_window).mean()
        dxy_std = dxy_momentum.rolling(window=rolling_window).std()
        features['dxy_momentum_zscore'] = (dxy_momentum - dxy_mean) / dxy_std
    
    # 4. Cross-asset correlation (equity vs FX, 60-day rolling)
    equity_cols = [col for col in combined_df.columns if col in ['SPY', 'QQQ', 'IWM']]
    fx_cols = [col for col in combined_df.columns if '=X' in col]
    
    if len(equity_cols) > 0 and len(fx_cols) > 0:
        equity_returns = combined_df[equity_cols].pct_change().mean(axis=1)
        fx_returns = combined_df[fx_cols].pct_change().mean(axis=1)
        
        # Rolling correlation
        correlation = []
        for i in range(len(equity_returns)):
            if i < rolling_window:
                correlation.append(np.nan)
            else:
                window_equity = equity_returns.iloc[i-rolling_window:i]
                window_fx = fx_returns.iloc[i-rolling_window:i]
                corr = window_equity.corr(window_fx)
                correlation.append(corr)
        
        features['cross_asset_correlation'] = correlation
    
    # 5-6. Raw signals (already normalized by design)
    features['equity_momentum'] = signals_df['equity_momentum']
    features['fx_carry'] = signals_df['fx_carry']
    
    # Drop rows with insufficient history
    features = features.dropna()
    
    return features

# Engineer features
features_df = engineer_features(combined_df, signals_df, rolling_window=60)

print("Feature engineering complete!")
print(f"\nFeatures shape: {features_df.shape}")
print(f"Features: {list(features_df.columns)}")
print(f"\nFeature statistics:")
print(features_df.describe())

# Plot features
fig, axes = plt.subplots(len(features_df.columns), 1, figsize=(14, 3*len(features_df.columns)), sharex=True)

for idx, col in enumerate(features_df.columns):
    axes[idx].plot(features_df.index, features_df[col], linewidth=1, alpha=0.7)
    axes[idx].axhline(y=0, color='r', linestyle='--', alpha=0.3)
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Date', fontsize=12)
plt.suptitle('Regime Detection Features', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../reports/figures/features.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Regime Detection

Fit Gaussian HMM to detect market regimes (risk-on vs risk-off).

In [ ]:
def detect_regimes(features_df, n_states=2, n_iter=100, random_state=42):
    """
    Detect market regimes using Gaussian HMM.
    
    Returns:
        DataFrame with regime labels and probabilities
    """
    # Prepare data for HMM (need numpy array)
    X = features_df.values
    
    # Standardize features (HMM works better with standardized data)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Fit HMM
    print(f"Fitting HMM with {n_states} states...")
    model = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=n_iter,
        random_state=random_state
    )
    model.fit(X_scaled)
    
    # Predict regimes
    regime_labels = model.predict(X_scaled)
    regime_probs = model.predict_proba(X_scaled)
    
    # Create results DataFrame
    results = pd.DataFrame({
        'regime': regime_labels,
        'regime_prob_0': regime_probs[:, 0],
        'regime_prob_1': regime_probs[:, 1] if n_states > 1 else None
    }, index=features_df.index)
    
    if n_states > 1:
        results['regime_prob_1'] = regime_probs[:, 1]
    
    # Calculate regime statistics
    print(f"\nRegime distribution:")
    regime_counts = pd.Series(regime_labels).value_counts().sort_index()
    for regime, count in regime_counts.items():
        pct = count / len(regime_labels) * 100
        print(f"  Regime {regime}: {count} days ({pct:.1f}%)")
    
    # Interpret regimes based on feature means
    print(f"\nRegime characteristics:")
    for regime in range(n_states):
        regime_mask = regime_labels == regime
        regime_features = features_df[regime_mask].mean()
        print(f"\n  Regime {regime} (mean features):")
        for feat, val in regime_features.items():
            print(f"    {feat}: {val:.3f}")
    
    return results, model, scaler

# Detect regimes
regime_results, hmm_model, feature_scaler = detect_regimes(
    features_df,
    n_states=config['regime']['n_states'],
    n_iter=config['regime']['n_iter'],
    random_state=config['regime']['random_state']
)

# Plot regime transitions
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Regime labels
colors = ['green', 'red', 'blue', 'orange']
for regime in range(config['regime']['n_states']):
    mask = regime_results['regime'] == regime
    axes[0].scatter(regime_results.index[mask], regime_results['regime'][mask], 
                    c=colors[regime], label=f'Regime {regime}', alpha=0.6, s=10)

axes[0].set_ylabel('Regime', fontsize=12)
axes[0].set_title('Regime Transitions Over Time', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yticks(range(config['regime']['n_states']))

# Regime probabilities
if config['regime']['n_states'] == 2:
    axes[1].fill_between(regime_results.index, 0, regime_results['regime_prob_0'], 
                        alpha=0.5, label='Regime 0 Prob', color='green')
    axes[1].fill_between(regime_results.index, regime_results['regime_prob_0'], 1, 
                        alpha=0.5, label='Regime 1 Prob', color='red')
else:
    for regime in range(config['regime']['n_states']):
        axes[1].plot(regime_results.index, regime_results[f'regime_prob_{regime}'], 
                    label=f'Regime {regime} Prob', linewidth=1.5, alpha=0.7)

axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_title('Regime Probabilities', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/figures/regimes.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Backtesting

Compare 4 strategies:
1. Static Equity Momentum (100% equity)
2. Static FX Carry (100% FX)
3. Static 50/50 (equal weight)
4. Dynamic Regime-Based (80/20 or 20/80 based on regime)

In [ ]:
def calculate_strategy_returns(signals_df, regime_labels, initial_capital=100000, 
                               transaction_cost_bps=5, slippage_bps=2):
    """
    Calculate returns for different strategies.
    
    Strategies:
    1. Static Equity: 100% equity momentum
    2. Static FX: 100% FX carry
    3. Static 50/50: Equal weight
    4. Dynamic: Regime-based allocation
    """
    # Align signals and regimes
    aligned_data = pd.DataFrame({
        'equity_momentum': signals_df['equity_momentum'],
        'fx_carry': signals_df['fx_carry'],
        'regime': regime_labels
    }).dropna()
    
    # Calculate daily returns from signals (simplified: use signal as return proxy)
    # In practice, you'd use actual asset returns, but for MVP we'll use signal strength
    equity_returns = aligned_data['equity_momentum'] / 100  # Convert to return-like
    fx_returns = aligned_data['fx_carry'] / 100
    
    # Normalize signals to be more return-like (use rolling volatility)
    equity_vol = equity_returns.rolling(60).std()
    fx_vol = fx_returns.rolling(60).std()
    
    # Scale signals by volatility to get more realistic returns
    equity_returns_scaled = (equity_returns / equity_vol) * 0.15  # Target 15% annual vol
    fx_returns_scaled = (fx_returns / fx_vol) * 0.10  # Target 10% annual vol
    
    # Fill NaN with 0
    equity_returns_scaled = equity_returns_scaled.fillna(0)
    fx_returns_scaled = fx_returns_scaled.fillna(0)
    
    # Strategy 1: Static Equity Momentum
    strategy1_returns = equity_returns_scaled.copy()
    
    # Strategy 2: Static FX Carry
    strategy2_returns = fx_returns_scaled.copy()
    
    # Strategy 3: Static 50/50
    strategy3_returns = 0.5 * equity_returns_scaled + 0.5 * fx_returns_scaled
    
    # Strategy 4: Dynamic Regime-Based
    # Regime 0 (risk-on): 80% equity, 20% FX
    # Regime 1 (risk-off): 20% equity, 80% FX
    strategy4_returns = pd.Series(index=aligned_data.index, dtype=float)
    for i, regime in enumerate(aligned_data['regime']):
        if regime == 0:  # Risk-on
            strategy4_returns.iloc[i] = 0.8 * equity_returns_scaled.iloc[i] + 0.2 * fx_returns_scaled.iloc[i]
        else:  # Risk-off
            strategy4_returns.iloc[i] = 0.2 * equity_returns_scaled.iloc[i] + 0.8 * fx_returns_scaled.iloc[i]
    
    # Apply transaction costs (simplified: assume rebalancing every day)
    # Cost = transaction_cost_bps + slippage_bps
    total_cost = (transaction_cost_bps + slippage_bps) / 10000
    
    # For dynamic strategy, apply cost when regime changes
    regime_changes = aligned_data['regime'].diff() != 0
    strategy4_returns[regime_changes] -= total_cost
    
    # Calculate cumulative returns
    strategies = {
        'Static Equity Momentum': strategy1_returns,
        'Static FX Carry': strategy2_returns,
        'Static 50/50': strategy3_returns,
        'Dynamic Regime-Based': strategy4_returns
    }
    
    cumulative_returns = {}
    equity_curves = {}
    
    for name, returns in strategies.items():
        cumulative = (1 + returns).cumprod()
        equity_curves[name] = initial_capital * cumulative
        cumulative_returns[name] = cumulative
    
    return strategies, cumulative_returns, equity_curves, aligned_data

# Run backtest
strategies, cum_returns, equity_curves, backtest_data = calculate_strategy_returns(
    signals_df,
    regime_results['regime'],
    initial_capital=config['backtest']['initial_capital'],
    transaction_cost_bps=config['backtest']['transaction_cost_bps'],
    slippage_bps=config['backtest']['slippage_bps']
)

print("Backtest complete!")
print(f"\nBacktest period: {backtest_data.index[0].date()} to {backtest_data.index[-1].date()}")
print(f"Total days: {len(backtest_data)}")


In [ ]:
# Calculate performance metrics
def calculate_metrics(returns_series, equity_curve, name):
    """Calculate performance metrics for a strategy."""
    total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0] - 1) * 100
    annual_return = ((equity_curve.iloc[-1] / equity_curve.iloc[0]) ** (252 / len(returns_series)) - 1) * 100
    
    # Volatility (annualized)
    volatility = returns_series.std() * np.sqrt(252) * 100
    
    # Sharpe ratio (assuming risk-free rate = 0)
    sharpe = (annual_return / 100) / (volatility / 100) if volatility > 0 else 0
    
    # Max drawdown
    running_max = equity_curve.expanding().max()
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = drawdown.min() * 100
    
    # Win rate (positive days)
    win_rate = (returns_series > 0).sum() / len(returns_series) * 100
    
    return {
        'Strategy': name,
        'Total Return (%)': total_return,
        'Annual Return (%)': annual_return,
        'Volatility (%)': volatility,
        'Sharpe Ratio': sharpe,
        'Max Drawdown (%)': max_drawdown,
        'Win Rate (%)': win_rate
    }

# Calculate metrics for all strategies
metrics_list = []
for name, returns in strategies.items():
    metrics = calculate_metrics(returns, equity_curves[name], name)
    metrics_list.append(metrics)

metrics_df = pd.DataFrame(metrics_list)
print("\n" + "="*80)
print("PERFORMANCE METRICS")
print("="*80)
print(metrics_df.to_string(index=False))

# Plot equity curves
fig, ax = plt.subplots(figsize=(14, 8))

for name, equity_curve in equity_curves.items():
    ax.plot(equity_curve.index, equity_curve.values, label=name, linewidth=2, alpha=0.8)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Equity Curves - Strategy Comparison', fontsize=16, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=config['backtest']['initial_capital'], color='black', linestyle='--', alpha=0.5, label='Initial Capital')

plt.tight_layout()
plt.savefig('../reports/figures/equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Feature Analysis by Regime

Analyze how features differ across regimes.

In [ ]:
# Merge features with regime labels
features_with_regime = features_df.copy()
features_with_regime['regime'] = regime_results['regime']

# Plot feature distributions by regime
n_features = len(features_df.columns)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
axes = axes.flatten() if n_features > 1 else [axes]

for idx, feature in enumerate(features_df.columns):
    ax = axes[idx]
    
    for regime in range(config['regime']['n_states']):
        regime_data = features_with_regime[features_with_regime['regime'] == regime][feature]
        ax.hist(regime_data, bins=50, alpha=0.6, label=f'Regime {regime}', 
               color=colors[regime], density=True)
    
    ax.set_xlabel(feature, fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{feature} Distribution by Regime', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(n_features, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics by regime
print("\n" + "="*80)
print("FEATURE STATISTICS BY REGIME")
print("="*80)
for regime in range(config['regime']['n_states']):
    print(f"\nRegime {regime}:")
    regime_features = features_with_regime[features_with_regime['regime'] == regime][features_df.columns]
    print(regime_features.describe())


## 7. Summary & Next Steps

### Key Findings
- Regime detection successfully identified distinct market states
- Dynamic regime-based allocation shows [results from metrics]
- [Add key insights here]

### MVP Limitations
- No walk-forward optimization (single train/test split)
- Simplified FX carry (momentum proxy, not true interest rate differential)
- Fixed transaction costs
- No position sizing optimization

### Next Steps
- Walk-forward validation
- True FX carry calculation (interest rate data)
- Dynamic position sizing (Kelly, risk parity)
- Alternative regime models (Markov-switching, LSTM)